# Run the TRF model

This notebook fits the temporal response function model to GLM kernels. It uses quick optimizer settings so readers can see parameters and plots immediately.

## 1. Setup

The project TRF model fits pre-event and post-event ramp shapes to each neuron's GLM kernel. Parameters are `[b, a, m, r, s]`: baseline, amplitude, ramp location, temporal width, and decay/time-scale.

In [ ]:
from pathlib import Path
import os

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
os.chdir(repo)

H5_PATH = repo / 'results_pack' / 'YH21SC' / 'V1_random_01' / 'V1_random_01.h5'
N_NEURONS = 8
KERNEL_WIN_MS = [-1500, 1500]
MAX_ITER = 120
N_INITS = 2

print(H5_PATH)
print('exists:', H5_PATH.exists())

## 2. First fit a small GLM kernel set

The TRF model is fit to `kernel_all = (n_neurons, n_kernel_times)`, not directly to raw `dff` traces. This cell repeats the compact GLM step from the previous notebook.

In [ ]:
import h5py
import numpy as np
from modeling.generative import run_glm_multi_sess

with h5py.File(H5_PATH, 'r') as f:
    labels = f['labels'][:N_NEURONS]
    dff = f['neural_trials']['dff'][:N_NEURONS]
    time = f['neural_trials']['time'][:]
    vol_time = f['neural_trials']['vol_time'][:]
    vol_stim_vis = f['neural_trials']['vol_stim_vis'][:]
    stim_labels = f['neural_trials']['stim_labels'][:]

dt = float(np.nanmedian(np.diff(time)))
l_idx = int(abs(KERNEL_WIN_MS[0]) / dt)
r_idx = int(abs(KERNEL_WIN_MS[1]) / dt)
kernel_time = np.arange(-l_idx, r_idx) * dt

kernel_all = run_glm_multi_sess(
    [dff], [time],
    [vol_time], [vol_stim_vis], [stim_labels],
    l_idx, r_idx)

print('kernel_all:', kernel_all.shape)
print('kernel_time:', kernel_time.shape)

## 3. Split kernels into pre and post windows

The project splits each GLM kernel at stimulus onset. The pre model sees the negative-time part; the post model sees the positive-time part.

In [ ]:
z_idx = np.searchsorted(kernel_time, 0)
neu_seq_l = kernel_all[:, :z_idx]
neu_time_l = kernel_time[:z_idx]
neu_seq_r = kernel_all[:, z_idx:]
neu_time_r = kernel_time[z_idx:]

print('pre kernel data:', neu_seq_l.shape, neu_time_l.shape)
print('post kernel data:', neu_seq_r.shape, neu_time_r.shape)

## 4. Fit quick TRF models

The full project helper `fit_trf_model` uses stronger settings (`max_iter=520`, `n_inits=5`). Here we call the same lower-level functions with smaller settings for a fast teaching run.

In [ ]:
import torch
from modeling.quantifications import (
    preprocess_data, optimize_params, reproduce_and_score,
    trf_model_pre, trf_model_post,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nsl, ntl, nsr, ntr = preprocess_data(neu_seq_l, neu_time_l, neu_seq_r, neu_time_r, device)

bounds_pre = [
    torch.tensor([-1, 1, -0.5, 1e-2, 1e-3], dtype=torch.float32, device=device),
    torch.tensor([ 1, 1, 0,    1.0, 1.5], dtype=torch.float32, device=device),
]
bounds_post = [
    torch.tensor([-1, 1, 0,    1e-2, 1e-3], dtype=torch.float32, device=device),
    torch.tensor([ 1, 1, 0.02, 0.1,  1], dtype=torch.float32, device=device),
]

params_pre = optimize_params(ntl, nsl, trf_model_pre, nsl.shape[0], bounds_pre, MAX_ITER, 1e-2, 1e-4, N_INITS, device)
pred_pre, r2_pre = reproduce_and_score(ntl, nsl, trf_model_pre, params_pre)

params_post = optimize_params(ntr, nsr, trf_model_post, nsr.shape[0], bounds_post, MAX_ITER, 1e-2, 1e-4, N_INITS, device)
pred_post, r2_post = reproduce_and_score(ntr, nsr, trf_model_post, params_post)

trf_param_pre = params_pre.detach().cpu().numpy()
trf_param_post = params_post.detach().cpu().numpy()
pred_pre = pred_pre.detach().cpu().numpy()
pred_post = pred_post.detach().cpu().numpy()
r2_pre = r2_pre.detach().cpu().numpy()
r2_post = r2_post.detach().cpu().numpy()

print('trf_param_pre:', trf_param_pre.shape)
print('pred_pre:', pred_pre.shape, 'r2_pre:', r2_pre.shape)
print('trf_param_post:', trf_param_post.shape)
print('pred_post:', pred_post.shape, 'r2_post:', r2_post.shape)

## 5. Print immediate results

Compare pre and post `R^2` for each neuron. Larger `R^2` means that side's ramp model better explains the GLM kernel shape.

In [ ]:
print('neuron  label  r2_pre  r2_post  best_side')
for i in range(len(r2_pre)):
    best = 'pre' if r2_pre[i] >= r2_post[i] else 'post'
    print(f'{i:6d} {labels[i]:6d} {r2_pre[i]:7.3f} {r2_post[i]:8.3f}  {best}')

print('\nmean r2_pre:', float(np.nanmean(r2_pre)))
print('mean r2_post:', float(np.nanmean(r2_post)))

## 6. Plot fitted examples

Black dots are normalized GLM kernel samples. Colored lines are TRF model predictions.

In [ ]:
import matplotlib.pyplot as plt

n_plot = min(4, N_NEURONS)
fig, axs = plt.subplots(n_plot, 2, figsize=(9, 2.2*n_plot), constrained_layout=True)
if n_plot == 1:
    axs = axs.reshape(1, 2)

ntl_np = ntl.detach().cpu().numpy()
ntr_np = ntr.detach().cpu().numpy()
nsl_np = nsl.detach().cpu().numpy()
nsr_np = nsr.detach().cpu().numpy()

for i in range(n_plot):
    axs[i, 0].scatter(ntl_np, nsl_np[i], s=8, color='black')
    axs[i, 0].plot(ntl_np, pred_pre[i], color='coral', lw=2)
    axs[i, 0].set_title(f'neuron {i} pre, R2={r2_pre[i]:.2f}')
    axs[i, 0].set_xlabel('normalized pre time')
    axs[i, 0].set_ylabel('normalized kernel')

    axs[i, 1].scatter(ntr_np, nsr_np[i], s=8, color='black')
    axs[i, 1].plot(ntr_np, pred_post[i], color='mediumseagreen', lw=2)
    axs[i, 1].set_title(f'neuron {i} post, R2={r2_post[i]:.2f}')
    axs[i, 1].set_xlabel('normalized post time')
    axs[i, 1].set_ylabel('normalized kernel')

plt.show()

## 7. Plot parameter summary

The parameter names are `b`, `a`, `m`, `r`, and `s`. The full report uses these values for ramp-type clustering.

In [ ]:
param_names = ['b', 'a', 'm', 'r', 's']
x = np.arange(len(param_names))

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
ax.plot(x, np.nanmean(trf_param_pre, axis=0), 'o-', color='coral', label='pre')
ax.plot(x, np.nanmean(trf_param_post, axis=0), 'o-', color='mediumseagreen', label='post')
ax.set_xticks(x)
ax.set_xticklabels(param_names)
ax.set_ylabel('mean fitted value')
ax.set_title('Mean TRF parameters across selected neurons')
ax.legend(frameon=False)
plt.show()

## 8. Full project call

For the full settings used by the project, call:

```python
from modeling.quantifications import fit_trf_model
results = fit_trf_model(neu_seq_l, neu_time_l, neu_seq_r, neu_time_r)
```

That returns `[trf_param_pre, pred_pre, r2_pre, trf_param_post, pred_post, r2_post]`.